In [3]:
#read labels
import pandas as pd
import os
from scipy.signal import savgol_filter  
import numpy as np
from tqdm import tqdm  # Import tqdm for progress display
from tensorflow.keras.preprocessing.sequence import pad_sequences
#create y
labels_path = r'./data/weak_supervision_results.csv'
df_labels = pd.read_csv(labels_path)
df_labels.set_index('ID',inplace=True)
y = df_labels['Prediction']
y.dropna(inplace=True)
y_binary = y>0

In [6]:
#read labels
import pandas as pd
import os
import numpy as np
from tqdm import tqdm  # Import tqdm for progress display
from tensorflow.keras.preprocessing.sequence import pad_sequences



#create x
tasks=['right', 'left']

df_combined = pd.DataFrame(columns=['values', 'ID'])
df_combined['ID'] = y.index
df_combined.set_index('ID', inplace=True)

for task in tasks:
    file =  rf'./data/distances-{task}/'
    segments_file = rf'./data/{task}_best_segments_and_indexes.csv'
    segments_file = pd.read_csv(segments_file).set_index('file_name')
    
    for root, dirs, filenames in os.walk(file):
        # Initialize progress bar
        filenames = [f for f in filenames if f.endswith('.csv')]
        with tqdm(total=len(filenames), desc=f"Processing {file}", unit="file") as pbar:
            for filename in filenames:
                file_path = os.path.join(root, filename)
                distances = pd.read_csv(file_path)[['Frame', 'Finger Distance', 'Finger Normalized Distance', 'Angular Distance', 'Wrist Coordinate']]
                if len(distances) < 5:
                    print(f"File {file} has less than 5 frames. Skipping...")
                    pbar.update(1)
                    continue
                
                #filter based on segments
                # if filename in segments_file.index:
                #     start,end,_ = segments_file.loc[filename].values
                #     distances = distances[(distances['Frame'] >= start) & (distances['Frame'] <= end)]
                    
                    
                normalized_distance = distances['Finger Normalized Distance']
                file_id = filename.split('_finger')[0]
                df_combined.loc[file_id, 'values'] = normalized_distance.to_numpy()
                pbar.update(1)
                
      

Processing ./data/distances-left/: 100%|██████████| 237/237 [00:00<00:00, 258.46file/s]


In [7]:
from tensorflow.keras.preprocessing.sequence import pad_sequences


#shuffle df_vcombined 
df_combined_shuffled = df_combined.copy().sample(frac=1, random_state=42)
# Select valid IDs
valid_ids = df_combined_shuffled['values'].dropna().index.intersection(y_binary.index)

# Extract sequences
sequences = df_combined_shuffled.loc[valid_ids, 'values'].tolist()

# Pad or truncate sequences to equal length
X_padded = pad_sequences(sequences, maxlen=450, dtype='float32', padding='post', truncating='post')


# Expand dims to match required shape: (samples, series_length, 1)
X = X_padded
y_binary = y_binary.loc[valid_ids]
y = y.loc[valid_ids]



# Check shapes
print(X.shape, y_binary.shape)


(424, 450) (424,)


## Binary Classifier

In [12]:
from sklearn.model_selection import train_test_split
from aeon.classification.deep_learning import TimeCNNClassifier
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Aeon classifiers
from aeon.classification.convolution_based import (
    MiniRocketClassifier,
    RocketClassifier,
    MultiRocketClassifier,
)
from aeon.classification.deep_learning import (
    InceptionTimeClassifier,
    ResNetClassifier,
    FCNClassifier,
)
from aeon.classification.dictionary_based import (
    BOSSEnsemble,
    WEASEL,
    TemporalDictionaryEnsemble,
)
from aeon.classification.distance_based import (
    KNeighborsTimeSeriesClassifier,
    ElasticEnsemble,
    ProximityForest,
)
from aeon.classification.shapelet_based import ShapeletTransformClassifier
from aeon.classification.feature_based import Catch22Classifier

X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.15, random_state=42, stratify=y_binary)
# Dictionary of classifiers
classifiers = {
     "MiniRocket": MiniRocketClassifier(),#0.88
    # "Rocket": RocketClassifier(),0.8
    # "MultiRocket": MultiRocketClassifier(),0.8
    # "InceptionTime": InceptionTimeClassifier(),
    # "ResNet": ResNetClassifier(),
    # "FCN": FCNClassifier(),
    # "BOSSEnsemble": BOSSEnsemble(),#0.77
    # "WEASEL": WEASEL(),#0.86
    # "TemporalDictionaryEnsemble": TemporalDictionaryEnsemble(),#0.81
    # "KNeighbors": KNeighborsTimeSeriesClassifier(),#0.75
    #"ElasticEnsemble": ElasticEnsemble(),
    #"ProximityForest": ProximityForest(),
    #"ShapeletTransform": ShapeletTransformClassifier(),
    #"Catch22Classifier": Catch22Classifier(),#0.84
}

# Evaluate each classifier
for name, clf in classifiers.items():
    print(f'\n=== Training {name} ===')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f'Classification Report for {name}:\n')
    print(classification_report(y_test, y_pred))


=== Training MiniRocket ===
Classification Report for MiniRocket:

              precision    recall  f1-score   support

       False       1.00      0.55      0.71        11
        True       0.91      1.00      0.95        53

    accuracy                           0.92        64
   macro avg       0.96      0.77      0.83        64
weighted avg       0.93      0.92      0.91        64



## Multi Class Classifier and regression

In [41]:
# test all classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from aeon.classification.deep_learning import TimeCNNClassifier
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Aeon classifiers
from aeon.classification.convolution_based import (
    MiniRocketClassifier,
    RocketClassifier,
    MultiRocketClassifier,
)
from aeon.classification.deep_learning import (
    InceptionTimeClassifier,
    ResNetClassifier,
    FCNClassifier,
)
from aeon.classification.dictionary_based import (
    BOSSEnsemble,
    WEASEL,
    TemporalDictionaryEnsemble,
)
from aeon.classification.distance_based import (
    KNeighborsTimeSeriesClassifier,
    ElasticEnsemble,
    ProximityForest,
)
from aeon.classification.shapelet_based import ShapeletTransformClassifier
from aeon.classification.feature_based import Catch22Classifier

y[y==4]=3
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y_binary)
# Dictionary of classifiers
classifiers = {
    "MiniRocket": MiniRocketClassifier(),
    "Rocket": RocketClassifier(),
    "MultiRocket": MultiRocketClassifier(),
    #"InceptionTime": InceptionTimeClassifier(),
    #"ResNet": ResNetClassifier(),
    #"FCN": FCNClassifier(),
    #"BOSSEnsemble": BOSSEnsemble(),
    #"WEASEL": WEASEL(),
    #"TemporalDictionaryEnsemble": TemporalDictionaryEnsemble(),
    #"KNeighbors": KNeighborsTimeSeriesClassifier(),
    #"ElasticEnsemble": ElasticEnsemble(),
    #"ProximityForest": ProximityForest(),
    #"ShapeletTransform": ShapeletTransformClassifier(),
    #"Catch22Classifier": Catch22Classifier(),
}

# Evaluate each classifier
for name, clf in classifiers.items():
    print(f'\n=== Training {name} ===')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f'Classification Report for {name}:\n')
    print(classification_report(y_test, y_pred))
    #show confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f'Confusion Matrix for {name}:\n')
    print(cm)


=== Training MiniRocket ===
Classification Report for MiniRocket:

              precision    recall  f1-score   support

         0.0       0.69      0.82      0.75        11
         1.0       0.50      0.30      0.38        10
         2.0       0.55      0.58      0.56        19
         3.0       0.72      0.75      0.73        24

    accuracy                           0.64        64
   macro avg       0.62      0.61      0.61        64
weighted avg       0.63      0.64      0.63        64

Confusion Matrix for MiniRocket:

[[ 9  0  1  1]
 [ 2  3  2  3]
 [ 2  3 11  3]
 [ 0  0  6 18]]

=== Training Rocket ===
Classification Report for Rocket:

              precision    recall  f1-score   support

         0.0       0.64      0.64      0.64        11
         1.0       0.25      0.30      0.27        10
         2.0       0.42      0.42      0.42        19
         3.0       0.55      0.50      0.52        24

    accuracy                           0.47        64
   macro avg    

In [67]:
#labels Analysis

df_experts = df_labels[['KW','MG','SA','WM']]
df_experts.dropna(how='all',inplace=True)

df_experts['number_of_experts'] = df_experts.notnull().sum(axis=1)
df_experts['majority_vote'] = df_experts[['KW','MG','SA','WM']].mode(axis=1)[0]
df_experts = df_experts[df_experts['number_of_experts']>1]

y = df_experts['majority_vote']
y_binary = y>0

# Select valid IDs
valid_ids = df_combined['values'].dropna().index.intersection(y_binary.index)


# Extract sequences
sequences = df_combined.loc[valid_ids, 'values'].tolist()

# Pad or truncate sequences to equal length
X_padded = pad_sequences(sequences, maxlen=500, dtype='float32', padding='post', truncating='post')


# Expand dims to match required shape: (samples, series_length, 1)
X = X_padded
y_binary = y_binary.loc[valid_ids]
y = y.loc[valid_ids]

# Check shapes
print(X.shape, y_binary.shape)



In [ ]:
#Add diff and diff2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from aeon.classification.convolution_based import MiniRocketClassifier, MultiRocketHydraClassifier,RocketClassifier
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter  
# Time series augmentation library
from tsaug import TimeWarp, AddNoise, Drift, Crop

def smooth_signal(signal, window_length=20, polyorder=2):
    if len(signal) < window_length:
        return signal
    return savgol_filter(signal, window_length=window_length, polyorder=polyorder)

# Define a function to calculate differences and apply Gaussian smoothing
def calculate_diffs_and_smooth(X):
    n_samples, series_length = X.shape

    # Smooth original
    X_smooth = smooth_signal(X)

    # First difference (same length using prepend)
    X_diff = np.diff(X, axis=1, prepend=X[:, [0]])
    X_diff_smooth = gaussian_filter1d(X_diff, sigma=1, axis=1)

    # Second difference (same length using prepend again)
    X_diff2 = np.diff(X_diff, axis=1, prepend=X_diff[:, [0]])
    X_diff2_smooth = gaussian_filter1d(X_diff2, sigma=1, axis=1)
    
    #return X
    return np.stack([X, X_diff], axis=1)

# Assuming X and y are already defined and processed as per your initial setup

# Perform class label adjustment and filtering (as per your original code)
y = np.where(y == 4, 3, y)  # Change label 4 to 3
mask = (y == 0) | (y == 1) | (y == 2) | (y == 3)
y_np_comp = y[mask]
X_comp = X[mask]

# Initialize StratifiedKFold for cross-validation
kf = StratifiedKFold(n_splits=8, shuffle=True, random_state=10)
scores = []
maes = []

print("\n=== Manual Cross-Validation for MiniRocketClassifier + Augmentation ===")

# Iterate over folds
for fold, (train_index, test_index) in enumerate(kf.split(X_comp, y_np_comp), 1):
    X_train, X_test = X_comp[train_index], X_comp[test_index]
    y_train, y_test = y_np_comp[train_index], y_np_comp[test_index]

    # Preprocess training and testing data with differences and smoothing
    X_train_augmented = calculate_diffs_and_smooth(X_train)
    X_test_augmented = calculate_diffs_and_smooth(X_test)

    # Train classifier
    clf = MiniRocketClassifier(random_state=42,n_kernels=12000)
    clf.fit(X_train_augmented, y_train)
    y_pred = clf.predict(X_test_augmented)

    print(f"\n--- Fold {fold} ---")
    #print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    print(f"Confusion Matrix:\n{cm}")

    accuracy = np.mean(y_pred == y_test)
    scores.append(accuracy)
    #print MAE and MSE
    print(f"Mean Absolute Error: {np.mean(np.abs(y_pred - y_test)):.4f}")
    print(f"Mean Squared Error: {np.mean((y_pred - y_test)**2):.4f}")
    maes.append(np.mean(np.abs(y_pred - y_test)))

print(f"\nCross-Validation Scores: {scores}")
print(f"Mean Accuracy: {np.mean(scores):.4f}")
print(f"Mean Absolute Error: {np.mean(maes):.4f}")



=== Manual Cross-Validation for MiniRocketClassifier + Augmentation ===

--- Fold 1 ---
Confusion Matrix:
[[ 3  5  0  1]
 [ 3  5  1  1]
 [ 1  3 11  3]
 [ 0  0  2 14]]

--- Fold 2 ---
Confusion Matrix:
[[ 2  6  0  1]
 [ 0  3  6  1]
 [ 0  3 11  4]
 [ 0  0  1 15]]

--- Fold 3 ---
Confusion Matrix:
[[ 7  2  0  0]
 [ 3  3  3  1]
 [ 2  1 11  4]
 [ 1  0  1 14]]

--- Fold 4 ---
Confusion Matrix:
[[ 4  2  3  0]
 [ 1  3  5  1]
 [ 0  3 12  3]
 [ 0  0  1 15]]

--- Fold 5 ---
Confusion Matrix:
[[ 8  0  2  0]
 [ 2  3  4  1]
 [ 0  3 10  4]
 [ 1  0  2 13]]

--- Fold 6 ---
Confusion Matrix:
[[ 5  1  2  1]
 [ 2  0  7  2]
 [ 2  1  9  5]
 [ 1  1  4 10]]

--- Fold 7 ---
Confusion Matrix:
[[5 2 2 0]
 [3 5 2 1]
 [0 5 9 3]
 [0 2 5 9]]

--- Fold 8 ---
Confusion Matrix:
[[ 4  3  0  2]
 [ 5  3  3  0]
 [ 1  0 10  7]
 [ 0  0  2 13]]

Cross-Validation Scores: [0.6226415094339622, 0.5849056603773585, 0.660377358490566, 0.6415094339622641, 0.6415094339622641, 0.4528301886792453, 0.5283018867924528, 0.566037735849056

In [153]:
# 3classifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from aeon.classification.convolution_based import MultiRocketClassifier
import numpy as np
from scipy.stats import mode

# Select only samples from classes 1, 2, and 3
mask = (y == 1) | (y == 2) | (y == 3)
y_np_comp = y[mask]
X_comp = X[mask]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

print("\n=== Cross-Validation with 3 Binary Classifiers + Majority Voting ===")

for fold, (train_index, test_index) in enumerate(kf.split(X_comp, y_np_comp), 1):
    X_train, X_test = X_comp[train_index], X_comp[test_index]
    y_train, y_test = y_np_comp[train_index], y_np_comp[test_index]

    # Prepare data for each binary classification
    def prepare_binary(X, y, class_a, class_b):
        mask = (y == class_a) | (y == class_b)
        X_bin = X[mask]
        y_bin = y[mask]
        y_bin = np.where(y_bin == class_a, 0, 1)
        return X_bin, y_bin

    # 1 vs 3
    X_13, y_13 = prepare_binary(X_train, y_train, 1, 3)
    clf_13 = MultiRocketClassifier(random_state=42, n_jobs=8, class_weight='balanced')
    clf_13.fit(X_13, y_13)

    # 2 vs 3
    X_23, y_23 = prepare_binary(X_train, y_train, 2, 3)
    clf_23 = MultiRocketClassifier(random_state=42, n_jobs=8, class_weight='balanced')
    clf_23.fit(X_23, y_23)

    # 1 vs 2
    X_12, y_12 = prepare_binary(X_train, y_train, 1, 2)
    clf_12 = MultiRocketClassifier(random_state=42, n_jobs=8, class_weight='balanced')
    clf_12.fit(X_12, y_12)

    # Predict with each classifier on the full test set
    def binary_predict(clf, class_a, class_b, X_test):
        pred_bin = clf.predict(X_test)
        return np.where(pred_bin == 0, class_a, class_b)

    pred_13 = binary_predict(clf_13, 1, 3, X_test)
    pred_23 = binary_predict(clf_23, 2, 3, X_test)
    pred_12 = binary_predict(clf_12, 1, 2, X_test)

    # Majority voting
    preds = np.vstack([pred_13, pred_23, pred_12])
    y_pred_majority = mode(preds, axis=0).mode

    # Evaluation
    print(f"\n--- Fold {fold} ---")
    print(classification_report(y_test, y_pred_majority))
    cm = confusion_matrix(y_test, y_pred_majority)
    print(f"Confusion Matrix:\n{cm}")

    accuracy = np.mean(y_pred_majority == y_test)
    scores.append(accuracy)

print(f"\nCross-Validation Scores: {scores}")
print(f"Mean Accuracy: {np.mean(scores):.4f}")


In [ ]:
#regression
from aeon.regression.convolution_based import MiniRocketRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, classification_report, confusion_matrix
import numpy as np

# Merge class 4 into class 3
y_np = np.where(y == 4, 3, y)

# Initialize cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
mse_scores = []
accuracy_scores = []

print("\n=== Manual Cross-Validation for MiniRocketRegressor ===")

for fold, (train_index, test_index) in enumerate(kf.split(X, y_np), 1):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y_np[train_index], y_np[test_index]
    
    reg = MiniRocketRegressor()
    reg.fit(X_train, y_train)
    y_pred_reg = reg.predict(X_test)

    # Calculate MSE
    mse = mean_squared_error(y_test, y_pred_reg)
    mse_scores.append(mse)

    # Convert regression output to class by rounding
    y_pred_class = np.round(y_pred_reg).astype(int)

    # Classification metrics
    accuracy = np.mean(y_pred_class == y_test)
    accuracy_scores.append(accuracy)

    print(f"\n--- Fold {fold} ---")
    print(f"MSE: {mse:.4f}")
    print(f"Accuracy (rounded): {accuracy:.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred_class))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_class))

print("\n=== Summary ===")
print(f"Mean MSE: {np.mean(mse_scores):.4f}")
print(f"Mean Classification Accuracy: {np.mean(accuracy_scores):.4f}")


In [77]:
#test parameters
from aeon.classification.convolution_based import MiniRocketClassifier
from sklearn.linear_model import RidgeClassifierCV, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Merge class 4 into class 3
y_np = np.where(y == 4, 3, y)

# Classifiers to try
estimators = {
    "RidgeClassifierCV": RidgeClassifierCV(),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42)
}

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, est in estimators.items():
    print(f"\n=== Testing MiniRocketClassifier with {name} ===")
    scores = []

    for fold, (train_index, test_index) in enumerate(kf.split(X, y_np), 1):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y_np[train_index], y_np[test_index]

        clf = MiniRocketClassifier(estimator=est)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        accuracy = np.mean(y_pred == y_test)
        scores.append(accuracy)

        print(f"\n--- Fold {fold} ---")
        print(f"Accuracy: {accuracy:.4f}")
        print("Classification Report:")
        print(classification_report(y_test, y_pred))
        print("Confusion Matrix:")
        print(confusion_matrix(y_test, y_pred))

    print(f"\nMean Accuracy for {name}: {np.mean(scores):.4f}")


# Using MAE

In [22]:
import pickle
import numpy as np
from aeon.classification.convolution_based import MiniRocketClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

def prepare_and_fit_minirocket_with_pca(X, selected_landmarks=[0,4,8], pca_components=2):
    """
    Prepare data, apply PCA on embeddings, and fit MiniRocketClassifier.

    Parameters:
        X: np.ndarray (shape: [samples, 1, series_length, landmarks, embedding_dim])
        y: np.ndarray (shape: [samples, ])
        selected_landmarks: landmarks indices to keep
        pca_components: number of PCA dimensions (default: 2)

    Returns:
        classifier: trained MiniRocketClassifier model
    """

    # Step 1: Remove redundant dimension
    X = np.squeeze(X, axis=1)  # (300, 400, 21, 16)

    # Step 2: Select specific landmarks
    X = X[:, :, selected_landmarks, :]  # (300, 400, 3, 16)

    # Step 3: Apply PCA on embeddings (last dimension)
    samples, series_length, n_landmarks, embed_dim = X.shape
    X_flat = X.reshape(-1, embed_dim)  # Combine samples, time, landmarks: (300*400*3, 16)

    # PCA fit-transform
    pca = PCA(n_components=pca_components)
    X_pca = pca.fit_transform(X_flat)  # (300*400*3, 2)

    # Restore original dimensions with reduced embeddings
    X_pca = X_pca.reshape(samples, series_length, n_landmarks, pca_components)  # (300, 400, 3, 2)

    # Step 4: Merge landmark and reduced embeddings into channels
    X_final = X_pca.reshape(samples, series_length, n_landmarks * pca_components)  # (300, 400, 6)

    # Transpose to (samples, channels, series_length)
    X_final = np.transpose(X_final, (0, 2, 1))  # (300, 6, 400)


    return X_final


data = np.load(r"G:\My Drive\Temporary Atefeh\stmae_embeddings_pd_5_old.npy",allow_pickle=True)
embeddings = data["embeddings"]
labels = data["labels"]
indexes = data["indexes"]
labels[labels==4] = 3

# Convert embeddings to format required by Aeon (n_samples, n_channels, series_length)
# Currently, embeddings might be (n_samples, embedding_dim). 
# MiniRocket expects at least 3D input (samples, channels, length).
# Here, we reshape assuming embeddings represent single-channel, and length equals embedding_dim.
embeddings = embeddings[:, np.newaxis, :]
embeddings = prepare_and_fit_minirocket_with_pca(embeddings,selected_landmarks=[0,4,8], pca_components=2)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.15, random_state=42, stratify=labels
)

# Initialize MiniRocketClassifier
classifier = MiniRocketClassifier()

# Fit classifier
classifier.fit(X_train, y_train)

# Predict on test set
y_pred = classifier.predict(X_test)

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

# Print evaluation results
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(report)


Accuracy: 0.4348
Classification Report:
              precision    recall  f1-score   support

         0.0       0.40      0.62      0.48        13
         1.0       0.40      0.15      0.22        13
         2.0       0.32      0.30      0.31        23
         3.0       0.59      0.65      0.62        20

    accuracy                           0.43        69
   macro avg       0.43      0.43      0.41        69
weighted avg       0.43      0.43      0.42        69

